# D6 — multi-head joint ablation (one compound at a time)

Spec (approved 2026-06-29): `docs/superpowers/specs/2026-06-29-multihead-lexical-ablation-design.md`.
Pre-registration: DECISIONS.md D6 entry — **commit before the first real forward pass**.
Claim under test, scoped: even the strongest lexical lead (screen reader) is
distributed across heads, not a localized circuit. Model: pythia-2.8b throughout.

**Upload to this Colab session:** this notebook + the `src/` folder (drag the folder
into the Files sidebar) + `results/pythia/pythia-2.8b-binding.csv` (into
`results/pythia/`, so Stage A reads the frozen sweep; out-of-sweep compounds like
stock_market run a fresh in-memory binding pass and say so in the ledger).

Flow: run Setup once → `sanity_check` once → edit Cell 1 (compound) → run Cell 2 →
repeat for screen_reader, alt_text, stock_market, semantic_html → zip and bring it
home. Model loads once per session; only the compound changes between visits.

## Setup

In [1]:
# Cell 00: Colab Stuff
import os

try:
    import google.colab
    IN_COLAB = True
    print("Running as a Colab notebook")
    os.system('pip install transformer_lens==2.17.0 --quiet')
    os.system('pip install transformers==4.57.6 --quiet')
    # NOTE: no 'pip install --upgrade numpy' here — on current Colab images it
    # desyncs numpy from the preinstalled scipy's compiled ABI
    # (ImportError: _center from numpy._core.umath). D8's Cell 00 still carries
    # the line above its own removal note; fixed here, fix D8 when convenient.
    print("Dependencies installed")
except ImportError:
    IN_COLAB = False

Running as a Colab notebook
Dependencies installed


In [1]:
# Cell 0: Project root + imports (expects the src/ FOLDER uploaded, not flat files)
import sys
from pathlib import Path
import torch

# Colab: files are in /content/ ; Local: notebook lives in notebooks/
if Path('/content/src').exists():
    PROJECT_ROOT = Path('/content')
else:
    PROJECT_ROOT = Path.cwd().parent

assert (PROJECT_ROOT / 'src' / 'd6_multihead_ablation.py').exists(), (
    'Upload the src/ folder next to this notebook (Files sidebar, drag the '
    'whole folder) — d6 needs binding, head_characterization, qk_ov, and '
    'd6_multihead_ablation together.')
sys.path.insert(0, str(PROJECT_ROOT))

from src.d6_multihead_ablation import run_d6, sanity_check, COMPOUNDS
print(f"Project root: {PROJECT_ROOT}")
print("Compounds in the panel:", list(COMPOUNDS))

Project root: /content
Compounds in the panel: ['screen_reader', 'alt_text', 'stock_market', 'semantic_html']


In [2]:
# Cell 0b: Load the model ONCE per session (single model, per spec)
from transformer_lens import HookedTransformer

if 'model' not in globals():
    device = ('cuda' if torch.cuda.is_available()
              else 'mps' if torch.backends.mps.is_available() else 'cpu')
    model = HookedTransformer.from_pretrained('pythia-2.8b', device=device)
    print(f"Loaded pythia-2.8b on {device} "
          f"({model.cfg.n_layers} layers x {model.cfg.n_heads} heads)")
else:
    print("Model already loaded — reusing.")

/usr/local/lib/python3.12/dist-packages/huggingface_hub/utils/_auth.py:104: UserWarning: 
Error while fetching `HF_TOKEN` secret value from your vault: 'Requesting secret HF_TOKEN timed out. Secrets can only be fetched when running from the Colab UI.'.
You are not authenticated with the Hugging Face Hub in this notebook.
If the error persists, please let us know by opening an issue on GitHub (https://github.com/huggingface/huggingface_hub/issues/new).
  warnings.warn(


config.json:   0%|          | 0.00/571 [00:00<?, ?B/s]

`torch_dtype` is deprecated! Use `dtype` instead!


model.safetensors:   0%|          | 0.00/5.68G [00:00<?, ?B/s]

tokenizer_config.json:   0%|          | 0.00/396 [00:00<?, ?B/s]

tokenizer.json: 0.00B [00:00, ?B/s]

special_tokens_map.json:   0%|          | 0.00/99.0 [00:00<?, ?B/s]

Loaded pretrained model pythia-2.8b into HookedTransformer
Loaded pythia-2.8b on cuda (32 layers x 32 heads)


In [3]:
# Cell 0c: Sanity asserts (spec: identity KL==0; plural==singular on L29/H7;
# late-layer set moves KL). Run ONCE before the first real ablation.
sanity_check(model)

sanity_check PASSED — identity KL 0.0; L29/H7 singular==plural (1e-05); late-layer set (L30, 8 heads) KL=3e-06


## One compound per visit
Edit Cell 1, run Cell 2. Order: `screen_reader` (primary — also runs the
bicycle-wheel negative control) → `alt_text` → `stock_market` → `semantic_html`.
Cell 3 shows the ledger. Reruns replace a compound's rows, never duplicate.

In [4]:
# Cell 1: Compound name variable
# screen_reader | alt_text | stock_market | semantic_html
compound_name = "screen_reader"

In [5]:
# Cell 2: Run D6 for this compound
# Stage A earns the lexical set (deep + selective + not sink/structural; the
# excluded heads become the labeled positive-control tail), Stage B runs the
# cumulative-knockout curve, both ledgers upsert. Seal untouched, no generation.
curves = run_d6(model, PROJECT_ROOT, compound_name)
curves

Stage A [screen_reader]: 18 deep candidates (min_layer=10, source: fresh in-memory pass (compound not in the frozen sweep))
  earned lexical set: 0 heads — (empty!)
  Stage A table → /content/results/pythia/pythia-2.8b-candidate-heads.csv
Stage B [screen_reader]: cumulative ablation over 5 heads (0 lexical + 5 tail)
  negative control: same set on 'A bicycle wheel is' at 'wheel'
  Stage B curves → /content/results/pythia/pythia-2.8b-multihead-ablation.csv

[screen_reader] peak KL — lexical set: nan | with structural tail: 0.000587
Flat lexical segment + rising tail = distributed result + working instrument, in one curve.


,compound,n_ablated,heads,newest_head,kl,segment,role
0,screen_reader,1,L29H7,L29H7,0.000012,structural_tail,primary
1,screen_reader,2,L29H7; L10H27,L10H27,0.000283,structural_tail,primary
2,screen_reader,3,L29H7; L10H27; L12H31,L12H31,0.000587,structural_tail,primary
3,screen_reader,4,L29H7; L10H27; L12H31; L27H10,L27H10,0.000577,structural_tail,primary
4,screen_reader,5,L29H7; L10H27; L12H31; L27H10; L11H7,L11H7,0.000508,structural_tail,primary
5,bicycle_wheel,1,L29H7,L29H7,0.000004,structural_tail,neg_control
6,bicycle_wheel,2,L29H7; L10H27,L10H27,0.000298,structural_tail,neg_control
7,bicycle_wheel,3,L29H7; L10H27; L12H31,L12H31,0.000594,structural_tail,neg_control
8,bicycle_wheel,4,L29H7; L10H27; L12H31; L27H10,L27H10,0.000665,structural_tail,neg_control
9,bicycle_wheel,5,L29H7; L10H27; L12H31; L27H10; L11H7,L11H7,0.000728,structural_tail,neg_control


In [7]:
import pandas as pd
cdf = pd.read_csv(PROJECT_ROOT / 'results/pythia/pythia-2.8b-candidate-heads.csv')
cols = ['layer','head','binding_score','own_score','max_other_score',
        'max_other_domain','selective','sink','structural','bos_attention',
        'attn_to_pos1','in_lexical_set']
print(cdf[cdf.compound=='screen_reader'][cols].to_string())

    layer  head  binding_score  own_score  max_other_score max_other_domain  selective   sink  structural  bos_attention  attn_to_pos1  in_lexical_set
0      29     7         0.9019     0.9019           0.2104          general       True   True       False         0.9117        0.0041           False
1      10    27         0.7677     0.7677           0.8383          general      False   True       False         0.5446        0.1136           False
2      12    31         0.5948     0.5948           0.7275          medical      False   True       False         0.5626        0.0811           False
3      27    10         0.3808     0.3808           0.2300          finance       True   True       False         0.8186        0.0123           False
4      20    12         0.3246     0.3246           0.3175            legal      False  False       False         0.2780        0.2068           False
5      11     7         0.3230     0.3230           0.2160          general      False   True 

In [6]:
# Cell 3: Ledger peek — which compounds exist so far, and the verdict shape
from pathlib import Path
import pandas as pd
led = PROJECT_ROOT / 'results/pythia/pythia-2.8b-multihead-ablation.csv'
if led.exists():
    df = pd.read_csv(led)
    print(df.groupby(['compound', 'role', 'segment'])['kl']
            .agg(['count', 'max']).round(6))
else:
    print('no ledger yet')
cand = PROJECT_ROOT / 'results/pythia/pythia-2.8b-candidate-heads.csv'
if cand.exists():
    cdf = pd.read_csv(cand)
    print('\nearned lexical sets:',
          cdf[cdf['in_lexical_set']].groupby('compound')
             .apply(lambda g: ', '.join(f"L{r.layer}H{r.head}"
                                        for r in g.itertuples()),
                    include_groups=False).to_dict())

                                           count       max
compound      role        segment                         
bicycle_wheel neg_control structural_tail      5  0.000728
screen_reader primary     structural_tail      5  0.000587

earned lexical sets: {'layer': {}, 'head': {}, 'binding_score': {}, 'induction': {}, 'prev_token': {}, 'dup_token': {}, 'type': {}, 'bos_attention': {}, 'attn_to_pos1': {}, 'own_score': {}, 'max_other_score': {}, 'max_other_domain': {}, 'selective': {}, 'sink': {}, 'structural': {}, 'in_lexical_set': {}, 'binding_source': {}}


## Bring it home

In [ ]:
# Cell 4: Zip + download (loss class: ephemeral /content)
import os
os.system('zip -j d6_results.zip '
          'results/pythia/pythia-2.8b-candidate-heads.csv '
          'results/pythia/pythia-2.8b-multihead-ablation.csv')
if IN_COLAB:
    from google.colab import files
    files.download('d6_results.zip')
else:
    print('Local run — results already live in results/pythia/, no zip needed.')